# Trend maps

_Cleaned copy (outputs stripped). Shared helpers in `functions.py`._


In [ ]:
import sys, os
# make this repo's functions.py importable from any subfolder
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline

In [ ]:
all_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5_all_trend.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
huss_tas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5_huss_tas_trend.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
rsds_rlds_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5_rsds_rlds.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
era5_winds_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5_winds.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
era5_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
jra55_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_jra55.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
uas_vas_trend  = xr.open_dataarray('/Users/isabelaconde/Desktop/data/grid_cell_trend/perturb_gridcell_trend_era5_uas_vas_trend.nc').sel(lat =slice(-20,20), lon = slice(110,300))*10*12
cm2_trends = xr.open_dataset('/Users/isabelaconde/Desktop/data/ACCESS_CM2/ACCESS-CM2_trends_1982-2017_gr.nc')
cm2_uas_vas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_gridcell_trend_cm2_uas_vas_trend.nc')

In [ ]:
cm2_trends

In [ ]:
noaa_sst = xr.open_dataset('/Users/isabelaconde/Desktop/data/conde2025/NOAA_sst.mon.mean.nc').sst.sel(time = slice('1982','2017')).sel(lat = slice(-80,80))
noaa_sst

def find_trend_values(dt):

    #Input: xarray array with time, lat, lon coordinates and a variable to calculate the trend
    #Output: xarray array with the trend values for each grid point
    dt = dt.sel(time=slice('1982','2017'))
    time, lat, lon = dt['time'].values, dt['lat'].values, dt['lon'].values
    sst_reshaped = dt.values.reshape(len(time), -1)
    degree = 1

    gradient_values = np.empty_like(sst_reshaped[0])
    for i in range(len(sst_reshaped[0])):
        y = sst_reshaped[:, i]
        coefficients = np.polyfit(np.arange(len(time)), y, degree)
        gradient_values[i] = coefficients[-2]

    gradient_values = gradient_values.reshape(lat.shape[0], lon.shape[0])
    gradient_da = xr.DataArray(gradient_values, coords={'lat': lat, 'lon': lon}, dims=['lat', 'lon'])
    return gradient_da 

noaa_trend = find_trend_values(noaa_sst.sel(lat =slice(-20,20), lon = slice(110,300)))*10*12
# noaa_trend

In [ ]:

das = [
    ("ERA5", era5_trend),
    ("JRA55", jra55_trend),
    ("uas & vas", uas_vas_trend),
    ("rsds & rlds", rsds_rlds_trend),
    ("huss & tas", huss_tas_trend),
    ("ERA5 winds", era5_winds_trend),
    ("all-ish", all_trend),
    ("NOAA OISST", noaa_trend),
    ("ACCESS-CM2", cm2_trends.sst_trend*10*12),
    ("cm2 uas & vas", cm2_uas_vas_trend)
]

# --- common colour limits (symmetric about 0) ---
stack = xr.concat([d for _, d in das], dim="panel")
v = float(np.nanmax(np.abs(stack.values)))
vmin, vmax = -0.35, 0.35

# --- map setup (same style as your example) ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 300, -20, 20]

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(4, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(3) for c in range(3)]

ims = []
for ax, (title, da) in zip(axes, das):
    im = ax.pcolormesh(da.lon, da.lat, da, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, transform=pc0)
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

for ax in axes[len(das):]:
    ax.set_visible(False)

gls = []
for ax in axes[:len(das)]:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    if (i % 3) != 0:
        gl.left_labels = False

cax = fig.add_axes([0.92, 0.20, 0.012, 0.60])
fig.colorbar(ims[0], cax=cax, label=r'$^{\circ}$C (10 yr)$^{-1}$')
plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.show()



In [ ]:
das = [
    ("ERA5", era5_trend),
    ("JRA55", jra55_trend),
    ("uas & vas", uas_vas_trend),
    ("rsds & rlds", rsds_rlds_trend),
    ("huss & tas", huss_tas_trend),
    ("ERA5 winds", era5_winds_trend),
    ("all-ish", all_trend),
    ("NOAA OISST", noaa_trend),
    ("ACCESS-CM2", cm2_trends.sst_trend * 10 * 12),
    ("cm2 uas & vas", cm2_uas_vas_trend*10*12),
]

vmin, vmax = -0.35, 0.35

proj = ccrs.PlateCarree(central_longitude=180)
pc0 = ccrs.PlateCarree()
extent = [110, 300, -20, 20]

fig = plt.figure(figsize=(16, 8))
gs = GridSpec(4, 3, figure=fig, wspace=0.03, hspace=-0.07)

# make enough axes for all panels
axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(4) for c in range(3)]

ims = []
gls = []

for i, (ax, (title, da)) in enumerate(zip(axes, das)):
    im = ax.pcolormesh(
        da.lon, da.lat, da,
        cmap="RdBu_r",
        vmin=vmin, vmax=vmax,
        transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)

    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False

    if (i % 3) != 0:
        gl.left_labels = False
    if i < len(das) - 3:
        gl.bottom_labels = False

    ims.append(im)
    gls.append(gl)

# hide unused axes
for ax in axes[len(das):]:
    ax.set_visible(False)

cax = fig.add_axes([0.92, 0.20, 0.012, 0.60])
fig.colorbar(ims[0], cax=cax, label=r"$^{\circ}$C (10 yr)$^{-1}$")

plt.show()

In [ ]:
import cartopy.feature as cfeature

fig, axes = plt.subplots(
    3, 1,
    figsize=(10, 8),
    subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
)

v = 0.3

plots = [
    (era5_trend,  -v, v, "RdBu_r", "ACCESS-OM2 forced with ERA5"),
    (jra55_trend, -v, v, "RdBu_r", "ACCESS-OM2 forced with JRA55-do"),
    (noaa_trend,  -v, v, "RdBu_r", "NOAA OISST V2")
]

for i, (ax, (da, vmin, vmax, cmap, title)) in enumerate(zip(axes, plots)):

    da.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        add_colorbar=False
    )

    ax.set_extent([110, 290, -20, 20], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor="white", zorder=10)
    ax.coastlines()

    ax.set_title(title, fontsize=14)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.5,
        linestyle="--"
    )

    gl.top_labels = False
    gl.left_labels = True
    gl.right_labels = False

    # gl.xlabel_style = {"size": 14}
    # gl.ylabel_style = {"size": 14}

    if i == 2:
        gl.bottom_labels = True
    else:
        gl.bottom_labels = False

# single colourbar
cbar = fig.colorbar(
    axes[0].collections[0],
    ax=axes,
    orientation="vertical",
    fraction=0.03,
    pad=0.02,
    shrink=0.97,
    aspect=40
)

cbar.set_label(
    r"C$^{\circ}$ (10 yr)$^{-1}$",
    fontsize=14
)

cbar.ax.tick_params(labelsize=14)

plt.show()